# Waste AI — Entraînement du modèle baseline
**MobileNetV3 + Transfer Learning sur TrashNet**

Lancez ce notebook sur Google Colab (GPU gratuit).

In [ ]:
# Installer les dépendances
!pip install torch torchvision matplotlib scikit-learn -q

# Télécharger TrashNet
!pip install kaggle -q
# Alternative directe :
!wget -q https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip
!unzip -q dataset-resized.zip -d data/

In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

DATA_DIR = 'data/dataset-resized'
BATCH_SIZE = 32
EPOCHS = 15
NUM_CLASSES = 6

In [ ]:
# Transformations
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Dataset complet
full_dataset = datasets.ImageFolder(DATA_DIR)
print('Classes :', full_dataset.classes)

# Split 80/20
n_train = int(0.8 * len(full_dataset))
n_val = len(full_dataset) - n_train
train_set, val_set = random_split(full_dataset, [n_train, n_val])
train_set.dataset.transform = train_tf
val_set.dataset.transform = val_tf

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train : {n_train} images | Val : {n_val} images')

In [ ]:
# Charger MobileNetV3 pré-entraîné
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

# Geler les couches de base
for param in model.features.parameters():
    param.requires_grad = False

# Remplacer la tête de classification
model.classifier[3] = nn.Linear(model.classifier[3].in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print('Modèle prêt.')

In [ ]:
# Boucle d'entraînement
history = {'train_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()

    # Validation
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total * 100
    avg_loss = total_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['val_acc'].append(acc)
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {acc:.1f}%')

In [ ]:
# Sauvegarder le modèle
import os
os.makedirs('checkpoints', exist_ok=True)
torch.save(model.state_dict(), 'checkpoints/waste_ai.pt')
print('Modèle sauvegardé -> checkpoints/waste_ai.pt')

# Courbes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss']); ax1.set_title('Loss')
ax2.plot(history['val_acc']); ax2.set_title('Val Accuracy (%)')
plt.tight_layout()
plt.savefig('checkpoints/training_curves.png')
plt.show()